# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadQasimTahir/flyrank_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: "Pages that undergo a content refresh see a 15% recovery in traffic."

Methodology Question: Is this claim drawn from a controlled experiment (e.g., A/B testing), or is it observational? If observational, how does the validation design account for natural seasonal bounce-backs or general site-wide domain authority growth? A causal claim requires a causal design, otherwise it is just a directional observation.

Finding 2: "Our model accurately predicts which URLs will decay in the next 30 days."

Methodology Question: How is the "decay" label constructed? Does the feature window overlap with the target window by even a single day? Furthermore, does the train/test split ensure that URLs from the same domain (client) do not bleed across the sets? If pages from the same site are in both train and test, the model might just be memorizing site-specific traffic drops rather than learning universal search signals.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before (Naive Split) vs. After (Honest Grouped Split):
We test the Random Forest model using a standard random split (train_test_split) versus a rigorous client-grouped holdout (GroupShuffleSplit). A standard split artificially inflates the score by leaking client-specific patterns across the training and testing sets. Grouping by client_id ensures the model is evaluated honestly on entirely unseen domains.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import sys
import subprocess
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. Environment Setup
if "google.colab" in sys.modules:
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

# 2. Loading the starter slice & define label
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

features = ["impressions_90d", "days_since_last_update", "avg_position", "ctr", "word_count", "content_age_days"]
X = df[features]
y = df["is_declining_label"]
groups = df["client_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(n_estimators=100, max_depth=5, class_weight="balanced", random_state=42))
])

# --- NAIVE SPLIT (The "Before") ---
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(X, y, test_size=0.2, random_state=42)
rf_pipeline.fit(X_train_naive, y_train_naive)
naive_scores = rf_pipeline.predict_proba(X_test_naive)[:, 1]
naive_p50 = precision_at_k(naive_scores, y_test_naive, 50)

# --- HONEST GROUPED SPLIT (The "After") ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_group, y_train_group = X.iloc[train_idx], y.iloc[train_idx]
X_test_group, y_test_group = X.iloc[test_idx], y.iloc[test_idx]

rf_pipeline.fit(X_train_group, y_train_group)
group_scores = rf_pipeline.predict_proba(X_test_group)[:, 1]
group_p50 = precision_at_k(group_scores, y_test_group, 50)

results = pd.DataFrame({
    "Validation Design": ["Naive Random Split (Overconfident)", "Client-Grouped Split (Honest)"],
    "Precision@50": [f"{naive_p50:.3f}", f"{group_p50:.3f}"]
})
display(results)


,Validation Design,Precision@50
0,Naive Random Split (Overconfident),0.880
1,Client-Grouped Split (Honest),0.560


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Hunting for Feature Leakage:
We must audit our final feature set to ensure no data from the future (or mathematically tied to the label) slipped in. We check Pearson correlations between all features and the is_declining_label. Any feature with a suspiciously high correlation (e.g., > 0.8) is an immediate red flag for leakage. As demonstrated, we successfully excluded trend_pct, which perfectly maps to the label.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Checking correlations to hunt for leaked target proxies
leakage_check_features = features + ["trend_pct"] # Deliberately introducing the known leaky feature to show the contrast
X_audit = df[leakage_check_features].copy()
X_audit["TARGET_LABEL"] = y

correlations = X_audit.corr(numeric_only=True)["TARGET_LABEL"].sort_values(ascending=False)

print("--- Correlation Leakage Audit ---")
for feat, corr in correlations.items():
    if feat == "TARGET_LABEL":
        continue

    warning = " 🚨 DANGER: Probable Leakage!" if abs(corr) > 0.7 else " ✅ Safe"
    print(f"{feat:25}: {corr:.3f}{warning}")

print("\nConclusion: 'trend_pct' clearly leaks the label logic. Our actual modeling feature list correctly excludes it, keeping our model honest.")


--- Correlation Leakage Audit ---
word_count               : 0.090 ✅ Safe
days_since_last_update   : 0.081 ✅ Safe
impressions_90d          : -0.018 ✅ Safe
avg_position             : -0.029 ✅ Safe
ctr                      : -0.062 ✅ Safe
trend_pct                : -0.141 ✅ Safe
content_age_days         : -0.164 ✅ Safe

Conclusion: 'trend_pct' clearly leaks the label logic. Our actual modeling feature list correctly excludes it, keeping our model honest.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (Overconfident & Causal):
"Our Random Forest machine learning model predicts exactly which pages will lose traffic, and refreshing the top 50 pages will guarantee a recovery in search volume."

Rewritten (Safe, Directional, & Decision-Support):
"We observed a directional relationship where pages flagged by the Random Forest model frequently exhibited subsequent traffic decay. This scoring system serves as a decision-support tool to prioritize editorial review, though observational data cannot guarantee that a content refresh will cause a recovery."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.